# Dengue Risk Forecasting with an Explainable Boosting Machine
## Reproducible analysis and model interpretation

**Goal:** Forecast next-week dengue case counts for each Sri Lankan district, then classify high/low outbreak risk using a district-specific threshold learned from the training period.

**Method:** Explainable Boosting Regressor (EBM), trained with a Poisson-deviance objective.

**Reproducibility:** Run this notebook from the `notebook/` directory inside the provided project. Install the dependencies first. Outputs are generated from the actual model rather than typed into the report.

**Important limitations:** The source pipeline linearly interpolates missing weeks using potentially future observations. This can introduce look-ahead bias. Reported test performance should be treated as provisional until a causal missing-data strategy is evaluated. The existing `cases_lag_1` is the *current observed week*, which is appropriate for next-week prediction but should not be described as the previous week.


## 1. Setup and load the original project functions
The core preprocessing, training and evaluation code is kept in `src/` to avoid duplicating business logic. The notebook makes each stage and its outputs visible for assessment.

In [ ]:
# If needed, run once in your notebook environment:
# %pip install -r ../src/requirements.txt

from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from data_prep import load_dataset, fill_missing_weeks, create_features, FEATURES, prepare_data
from model_training import train_model, load_model
from evaluation import (evaluate_regression, classify_risk_and_evaluate,
                        get_district_thresholds)
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score

DATASET = ROOT / "src" / "srilanka_dengue_weekly_by_district.csv"
MODEL = ROOT / "src" / "srilanka_dengue_ebm_model.pkl"
OUT = ROOT / "notebook" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
print("Dataset:", DATASET, "| exists:", DATASET.exists())
print("Model:", MODEL, "| exists:", MODEL.exists())


## 2. Dataset inspection
Inspect sample rows, missing values, date range, district count, and dengue case distribution before feature engineering.

In [ ]:
raw = load_dataset(DATASET)
display(raw.head())
print("Rows, columns:", raw.shape)
print("Districts:", raw["district"].nunique())
print("Date range:", raw["start.date"].min(), "to", raw["start.date"].max())
print("Missing values:\n", raw.isna().sum())
display(raw["cases"].describe().to_frame("cases"))


## 3. Preprocessing, feature engineering, and chronological split
Seasonality is encoded with sine/cosine terms. The four case-history columns and rolling mean summarize the recent trend. The target is next week's cases. The final target year is reserved for testing.

**Leakage warning:** `fill_missing_weeks()` currently uses linear interpolation and backward fill; those operations can use future data. This notebook preserves the supplied pipeline so results remain comparable, but they should not be represented as leakage-free.

In [ ]:
filled = fill_missing_weeks(raw)
feature_df, districts = create_features(filled)
test_year = int(feature_df["target_date"].dt.year.max())
train_df = feature_df.loc[feature_df["target_date"].dt.year < test_year].copy()
test_df = feature_df.loc[feature_df["target_date"].dt.year == test_year].copy()
X_train, y_train = train_df[FEATURES], train_df["target"]
X_test, y_test = test_df[FEATURES], test_df["target"]
print("Features:", FEATURES)
print("Train:", X_train.shape, "Test:", X_test.shape, "Test target year:", test_year)
display(X_train.head())
assert train_df["target_date"].max() < test_df["target_date"].min()


## 4. Load the saved EBM or retrain
The saved model is used by default to avoid unnecessarily long training. Set `RETRAIN=True` to regenerate it using the same training pipeline.

In [ ]:
RETRAIN = False
if RETRAIN:
    ebm = train_model(X_train, y_train, train_df, districts,
                      model_path=str(MODEL), risk_percentile=0.75)
model_data = load_model(str(MODEL))
ebm = model_data["model"]
print("Model:", type(ebm).__name__)
print("Model features:", model_data["features"])


## 5. Forecast evaluation
MAE and RMSE are in dengue-case units; smaller is better. R² compares the predictions with a mean-only baseline and may be negative.

In [ ]:
pred = np.asarray(ebm.predict(X_test))
regression = evaluate_regression(y_test, pred)
display(pd.DataFrame([regression]).round(4))
fig, ax = plt.subplots(figsize=(12, 4))
n = min(200, len(pred))
ax.plot(np.asarray(y_test)[:n], label="Actual")
ax.plot(pred[:n], label="Predicted")
ax.set(xlabel="Test observation", ylabel="Next-week cases",
       title="Actual versus predicted dengue cases")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "actual_vs_predicted.png", dpi=180)
plt.show()


## 6. Outbreak-risk evaluation
High risk is defined as next-week cases greater than or equal to the district's **training-period** 75th percentile. ROC-AUC is calculated using the continuous margin `predicted_cases - threshold`, not binary predictions.

In [ ]:
thresholds = get_district_thresholds(model_data, train_df, percentile=0.75)
risk_df, classification = classify_risk_and_evaluate(test_df, pred, thresholds)
display(pd.DataFrame([classification]).round(4))
cm = confusion_matrix(risk_df["actual_risk"], risk_df["predicted_risk"], labels=[0,1])
tn, fp, fn, tp = cm.ravel()
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")
ConfusionMatrixDisplay(cm, display_labels=["Low risk", "High risk"]).plot(cmap="Blues")
plt.title("Dengue risk confusion matrix")
plt.tight_layout()
plt.savefig(OUT / "confusion_matrix.png", dpi=180)
plt.show()

if risk_df["actual_risk"].nunique() == 2:
    fpr, tpr, _ = roc_curve(risk_df["actual_risk"], risk_df["risk_score"])
    fig, ax = plt.subplots(figsize=(6,5))
    ax.plot(fpr, tpr, label=f"EBM (AUC={classification['ROC-AUC']:.3f})")
    ax.plot([0,1], [0,1], "--", color="gray")
    ax.set(xlabel="False positive rate", ylabel="True positive rate", title="Risk ROC curve")
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "roc_curve.png", dpi=180)
    plt.show()


## 7. Global explainability: feature importance
EBM's built-in global explanation estimates how strongly each model term contributes to predictions. Importance is *associational*, not causal. An interaction term can appear alongside single features.

In [ ]:
global_exp = ebm.explain_global()
global_data = global_exp.data()
names = global_data.get("names", [])
scores = global_data.get("scores", [])
importance = (pd.DataFrame({"Term": names, "Importance": scores})
              .sort_values("Importance", ascending=False))
display(importance)
fig, ax = plt.subplots(figsize=(10, max(4, len(importance)*0.35)))
top = importance.head(20).iloc[::-1]
ax.barh(top["Term"], top["Importance"])
ax.set(xlabel="EBM global importance", title="Most influential model terms")
fig.tight_layout()
fig.savefig(OUT / "feature_importance.png", dpi=180, bbox_inches="tight")
plt.show()


## 8. EBM shape functions
These curves show how a **single model term** contributes to the model's prediction, holding other model terms fixed in the additive decomposition. With the Poisson objective, the vertical axis may be on the model's **link (log) scale**, so do **not** label it as a change in case counts. These are not automatically causal effects or conventional partial-dependence plots.

In [ ]:
# Plot selected one-dimensional terms (avoid pairwise interactions).
requested = ["cases_lag_1", "cases_lag_2", "cases_rolling_avg_4", "week_sin", "week_cos"]
for feature in requested:
    if feature not in list(ebm.term_names_):
        print("Not found:", feature)
        continue
    idx = list(ebm.term_names_).index(feature)
    term = global_exp.data(idx)
    x, y = term.get("names", []), term.get("scores", [])
    if len(x) != len(y):
        print("Skipping irregular term:", feature)
        continue
    fig, ax = plt.subplots(figsize=(8,3.5))
    ax.plot(x, y, marker=".", markersize=3)
    ax.set(xlabel=feature, ylabel="EBM term contribution (link scale)",
           title=f"EBM learned shape: {feature}")
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(OUT / f"shape_{feature}.png", dpi=180)
    plt.show()


## 9. Local explanation for an individual prediction
The EBM can explain which terms contribute to a particular district-week prediction. The output is an additive decomposition on the model link scale; use it to explain model behavior, not disease causation.

In [ ]:
local_exp = ebm.explain_local(X_test.iloc[:1], y_test.iloc[:1])
local = local_exp.data(0)
local_terms = pd.DataFrame({
    "Term": local.get("names", []),
    "Contribution": local.get("scores", [])
}).sort_values("Contribution", key=abs, ascending=False)
print("Observed next-week cases:", float(y_test.iloc[0]))
print("Predicted next-week cases:", float(pred[0]))
display(local_terms.head(15))


## 10. Export verified metrics for the report
The JSON below is generated from predictions on the actual test set. Use these values to populate the report. **Do not copy any metric from the existing report unless it agrees with these outputs.**

In [ ]:
metrics = {
    "test_year": test_year,
    "training_rows": len(X_train),
    "test_rows": len(X_test),
    "regression": {k: float(v) for k,v in regression.items()},
    "classification": {k: (float(v) if pd.notna(v) else None) for k,v in classification.items()},
    "confusion_matrix": {"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)},
    "top_terms": importance.head(10).to_dict(orient="records"),
    "limitations": ["Future-data leakage risk from interpolation/bfill",
                    "Associational explanations are not causal"]
}
with open(OUT / "evaluation_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))


## 11. Conclusions and limitations
Interpret the metrics generated above, particularly missed high-risk weeks (false negatives), which matter for outbreak-warning systems. Compare feature-importance rankings and shape functions with your hypotheses, but do not claim a particular ranking or seasonal peak without examining the plots. This model does not directly use rainfall, temperature, or population data. Before operational deployment, revisit missing-week imputation, validate with multiple forward-chaining splits, and monitor performance over time.